# Semiconductor Wafer Defect Classification (WM-811K)
Custom CNN vs ResNet-18, with class-weighted loss, geometric augmentation, full metrics and Grad-CAM.

**Dataset facts (from the uploaded archive):** 9 class folders (Center, Donut, Edge Local, Edge Ring, Local, Scratch, near full, none, random), ~100 images each, 32x32 RGB JPEG. Roughly 900 images in total, so the set is small and almost balanced. The notebook still computes class weights, so it stays correct if you swap in a larger or imbalanced version.

**Kaggle setup:** Add the dataset via *Add Input* and turn on the GPU (Settings, Accelerator). The data root is auto-detected.

In [ ]:
import os, glob, random, time, json, copy
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

# ---------------- CONFIG ----------------
CFG = dict(
    img_size   = 64,      # wafer maps are 32x32; upsampled (the paper also uses 64x64)
    batch_size = 64,
    epochs     = 60,
    lr         = 1e-3,
    weight_decay = 1e-4,
    patience   = 12,      # early stopping on val macro-F1
    seed       = 42,
    val_frac   = 0.15,
    test_frac  = 0.15,
    resnet_pretrained = False,   # True needs internet ON in Kaggle (ImageNet weights)
    drop_classes = [],           # e.g. ['near full','none'] to follow the paper's 7-class setup
)
OUT = Path('/kaggle/working') if Path('/kaggle').exists() else Path('./outputs')
OUT.mkdir(parents=True, exist_ok=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Load data and locate the dataset

In [ ]:
def find_root():
    search = ['/kaggle/input', '.', '/content']
    for base in search:
        for dp, dn, fn in os.walk(base):
            if os.path.basename(dp) == 'WM811k_Dataset':
                return dp
            # fallback: a folder whose subfolders contain images and look like classes
    raise FileNotFoundError('WM811k_Dataset folder not found. Check the dataset is attached.')

ROOT = find_root()
print('root:', ROOT)

rows = []
for cls in sorted(os.listdir(ROOT)):
    d = os.path.join(ROOT, cls)
    if not os.path.isdir(d): continue
    for f in os.listdir(d):
        if f.lower().endswith(('.jpg','.jpeg','.png')):
            rows.append((os.path.join(d, f), cls))
df = pd.DataFrame(rows, columns=['path','label'])
if CFG['drop_classes']:
    df = df[~df.label.isin(CFG['drop_classes'])].reset_index(drop=True)

CLASSES = sorted(df.label.unique())
cls2idx = {c:i for i,c in enumerate(CLASSES)}
df['y'] = df.label.map(cls2idx)
print(len(df), 'images,', len(CLASSES), 'classes')
print(df.label.value_counts())

## 2. Cleaning and visualisation

In [ ]:
# --- cleaning: unreadable files, size/mode consistency, exact duplicates ---
bad, sizes, hashes = [], {}, {}
for i, p in enumerate(df.path):
    try:
        with Image.open(p) as im:
            im.load()
            sizes[p] = (im.size, im.mode)
            hashes[p] = hash(np.asarray(im).tobytes())
    except Exception as e:
        bad.append(p)
print('unreadable:', len(bad))
print('size/mode combos:', pd.Series(sizes).astype(str).value_counts().to_dict())

dup = pd.Series(hashes)
dup_paths = dup[dup.duplicated(keep='first')].index.tolist()
print('exact duplicate images:', len(dup_paths))
df = df[~df.path.isin(bad + dup_paths)].reset_index(drop=True)
print('after cleaning:', len(df))

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 8, figsize=(12, 1.5*len(CLASSES)))
for r, c in enumerate(CLASSES):
    sub = df[df.label==c].sample(8, random_state=0)
    for k, p in enumerate(sub.path):
        axes[r,k].imshow(Image.open(p)); axes[r,k].axis('off')
        if k==0: axes[r,k].set_title(c, fontsize=9, loc='left')
plt.tight_layout(); plt.savefig(OUT/'samples.png', dpi=120); plt.show()

plt.figure(figsize=(7,3)); sns.countplot(data=df, y='label', order=CLASSES); plt.title('Class distribution')
plt.tight_layout(); plt.savefig(OUT/'class_dist.png', dpi=120); plt.show()

## 3. Split, preprocessing, augmentation, class weights
Stratified 70/15/15 split. Augmentation uses exact 90-degree rotations and flips, valid because a wafer defect label does not depend on orientation (the argument in the reference paper). A little Gaussian noise is added too.

In [ ]:
train_df, tmp = train_test_split(df, test_size=CFG['val_frac']+CFG['test_frac'],
                                 stratify=df.y, random_state=CFG['seed'])
val_df, test_df = train_test_split(tmp, test_size=CFG['test_frac']/(CFG['val_frac']+CFG['test_frac']),
                                   stratify=tmp.y, random_state=CFG['seed'])
print(len(train_df), len(val_df), len(test_df))

MEAN, STD = 0.5, 0.5   # simple scaling to [-1,1]; the maps have no natural ImageNet statistics

def load_tensor(path):
    im = Image.open(path).convert('RGB').resize((CFG['img_size'],)*2, Image.NEAREST)
    return torch.from_numpy(np.asarray(im)).permute(2,0,1).float().div(255)   # 3xHxW in [0,1]

class WaferDS(Dataset):
    def __init__(self, frame, train=False, fixed_tf=None):
        self.x = torch.stack([load_tensor(p) for p in frame.path])   # tiny dataset, cache in RAM
        self.y = torch.tensor(frame.y.values)
        self.train, self.fixed_tf = train, fixed_tf
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        x = self.x[i]
        if self.train:
            x = torch.rot90(x, random.randint(0,3), dims=(1,2))
            if random.random() < .5: x = x.flip(2)
            if random.random() < .5: x = x.flip(1)
            if random.random() < .3: x = (x + 0.02*torch.randn_like(x)).clamp(0,1)
        if self.fixed_tf: x = self.fixed_tf(x)
        return (x-MEAN)/STD, self.y[i]

train_ds, val_ds, test_ds = WaferDS(train_df, True), WaferDS(val_df), WaferDS(test_df)
mk = lambda ds, sh: DataLoader(ds, CFG['batch_size'], shuffle=sh, num_workers=2, pin_memory=True)
train_dl, val_dl, test_dl = mk(train_ds, True), mk(val_ds, False), mk(test_ds, False)

# class weights (inverse frequency, sample-weighted mean = 1) -> weighted cross-entropy
counts = np.bincount(train_df.y, minlength=len(CLASSES))
cw = counts.sum() / (len(CLASSES) * counts)
class_weights = torch.tensor(cw, dtype=torch.float32, device=device)
print(dict(zip(CLASSES, cw.round(3))))

## 4. Models: custom CNN and ResNet-18

In [ ]:
class CustomCNN(nn.Module):
    # 4 conv blocks (conv-BN-ReLU x2 + pool), global average pooling head
    def __init__(self, n_classes, ch=(32,64,128,256), p=0.4):
        super().__init__()
        layers, cin = [], 3
        for c in ch:
            layers += [nn.Conv2d(cin,c,3,padding=1,bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c,c,3,padding=1,bias=False),   nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            cin = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(p),
                                  nn.Linear(cin,128), nn.ReLU(inplace=True), nn.Dropout(p), nn.Linear(128,n_classes))
    def forward(self, x): return self.head(self.features(x))

def make_resnet18(n_classes, pretrained=False):
    w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    m = torchvision.models.resnet18(weights=w)
    if not pretrained:
        # small-image stem: 3x3 stride-1 conv and no maxpool (inputs are only 64x64)
        m.conv1 = nn.Conv2d(3,64,3,1,1,bias=False); m.maxpool = nn.Identity()
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    return m

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
n = len(CLASSES)
print('CustomCNN params:', n_params(CustomCNN(n)), '| ResNet-18 params:', n_params(make_resnet18(n)))

## 5. Training loop (weighted loss, AdamW + cosine LR, AMP, early stopping on val macro-F1)

In [ ]:
@torch.no_grad()
def predict(model, dl):
    model.eval(); P, Y, L = [], [], []
    for x, y in dl:
        lo = model(x.to(device)); P.append(lo.argmax(1).cpu()); Y.append(y); L.append(lo.cpu())
    return torch.cat(P).numpy(), torch.cat(Y).numpy(), torch.cat(L)

def macro_f1(y, p): return precision_recall_fscore_support(y, p, average='macro', zero_division=0)[2]

def train_model(model, name):
    model.to(device)
    crit = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['lr'], epochs=CFG['epochs'],
                                                steps_per_epoch=len(train_dl))
    scaler = torch.cuda.amp.GradScaler(enabled=device.type=='cuda')
    hist, best, best_state, bad = [], -1, None, 0
    t0 = time.time()
    for ep in range(CFG['epochs']):
        model.train(); tl, tc, tn = 0, 0, 0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type=='cuda'):
                out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item()*len(y); tc += (out.argmax(1)==y).sum().item(); tn += len(y)
        p, yv, lv = predict(model, val_dl)
        vl = F.cross_entropy(lv, torch.tensor(yv), weight=class_weights.cpu()).item()
        vf1, vacc = macro_f1(yv, p), accuracy_score(yv, p)
        hist.append(dict(epoch=ep+1, train_loss=tl/tn, train_acc=tc/tn, val_loss=vl, val_acc=vacc, val_f1=vf1))
        if vf1 > best:
            best, bad, best_state = vf1, 0, copy.deepcopy(model.state_dict())
        else:
            bad += 1
        if (ep+1) % 5 == 0 or ep == 0:
            print(f'[{name}] ep {ep+1:3d} | train loss {tl/tn:.3f} acc {tc/tn:.3f} | val loss {vl:.3f} acc {vacc:.3f} F1 {vf1:.3f}')
        if bad >= CFG['patience']:
            print(f'[{name}] early stop at epoch {ep+1}'); break
    model.load_state_dict(best_state)
    torch.save(best_state, OUT/f'{name}_best.pt')
    print(f'[{name}] best val macro-F1 {best:.4f}  ({time.time()-t0:.0f}s)')
    return model, pd.DataFrame(hist)

seed_everything(CFG['seed'])
cnn, h_cnn = train_model(CustomCNN(n), 'custom_cnn')
seed_everything(CFG['seed'])
res, h_res = train_model(make_resnet18(n, CFG['resnet_pretrained']), 'resnet18')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15,3.6))
for h, nme in [(h_cnn,'Custom CNN'), (h_res,'ResNet-18')]:
    ax[0].plot(h.epoch, h.train_loss, label=f'{nme} train'); ax[0].plot(h.epoch, h.val_loss, '--', label=f'{nme} val')
    ax[1].plot(h.epoch, h.train_acc, label=f'{nme} train'); ax[1].plot(h.epoch, h.val_acc, '--', label=f'{nme} val')
    ax[2].plot(h.epoch, h.val_f1, label=nme)
for a, t in zip(ax, ['Loss','Accuracy','Val macro-F1']): a.set_title(t); a.set_xlabel('epoch'); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT/'curves.png', dpi=120); plt.show()

## 6. Test-set evaluation: accuracy, precision, recall, macro-F1, confusion matrices

In [ ]:
results, preds = {}, {}
for name, m in [('Custom CNN', cnn), ('ResNet-18', res)]:
    p, y, _ = predict(m, test_dl); preds[name] = (p, y)
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average='macro', zero_division=0)
    results[name] = dict(Accuracy=accuracy_score(y,p), Precision_macro=pr, Recall_macro=rc, F1_macro=f1)
    print(f'===== {name} =====')
    print(classification_report(y, p, target_names=CLASSES, digits=3, zero_division=0))
summary = pd.DataFrame(results).T.round(4)
display(summary); summary.to_csv(OUT/'test_summary.csv')

fig, ax = plt.subplots(1, 2, figsize=(14,5.5))
for a, (name, (p, y)) in zip(ax, preds.items()):
    cm = confusion_matrix(y, p, labels=range(n))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=a, cbar=False)
    a.set_title(f'{name} (acc {results[name]["Accuracy"]:.3f})'); a.set_xlabel('Predicted'); a.set_ylabel('True')
    a.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.savefig(OUT/'confusion_matrices.png', dpi=120); plt.show()

### Rotation / flip robustness (out-of-distribution check, as in the reference paper)
The test set is evaluated again after rotating by 90/180/270 degrees and flipping. Training already used these augmentations, so this checks the effect of augmentation rather than architectural invariance. To reproduce the paper's baseline-vs-invariant comparison, retrain with augmentation turned off.

In [ ]:
tfs = {'original': lambda x: x,
       'rot90':  lambda x: torch.rot90(x,1,(1,2)), 'rot180': lambda x: torch.rot90(x,2,(1,2)),
       'rot270': lambda x: torch.rot90(x,3,(1,2)), 'flipH': lambda x: x.flip(2), 'flipV': lambda x: x.flip(1)}
rob = {}
for name, m in [('Custom CNN', cnn), ('ResNet-18', res)]:
    rob[name] = {}
    for t, f in tfs.items():
        ds = WaferDS(test_df, fixed_tf=f); p, y, _ = predict(m, DataLoader(ds, 128))
        rob[name][t] = macro_f1(y, p)
rob = pd.DataFrame(rob).round(4); display(rob); rob.to_csv(OUT/'robustness_f1.csv')

## 7. Grad-CAM

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.m, self.act, self.grad = model.eval(), None, None
        layer.register_forward_hook(lambda m,i,o: setattr(self,'act',o))
        layer.register_full_backward_hook(lambda m,gi,go: setattr(self,'grad',go[0]))
    def __call__(self, x, cls=None):
        x = x.unsqueeze(0).to(device).requires_grad_(True)
        out = self.m(x); cls = out.argmax(1).item() if cls is None else cls
        self.m.zero_grad(); out[0,cls].backward()
        w = self.grad.mean((2,3), keepdim=True)
        cam = F.relu((w*self.act).sum(1, keepdim=True))
        cam = F.interpolate(cam, x.shape[-2:], mode='bilinear', align_corners=False)[0,0]
        cam = (cam-cam.min())/(cam.max()-cam.min()+1e-8)
        prob = F.softmax(out,1)[0].detach().cpu()
        return cam.detach().cpu().numpy(), cls, prob

# target layers: last conv block of the CNN (the final ReLU before pooling), last residual block of ResNet
# inplace ReLU breaks backward hooks -> disable first (for both models)
for mdl in (cnn, res):
    for m in mdl.modules():
        if isinstance(m, nn.ReLU): m.inplace = False
cnn_layer = [m for m in cnn.features if isinstance(m, nn.ReLU)][-1]
cams = {'Custom CNN': GradCAM(cnn, cnn_layer), 'ResNet-18': GradCAM(res, res.layer4[-1])}

def show_gradcam(idxs, title, fname):
    fig, axes = plt.subplots(len(idxs), 5, figsize=(11, 2.2*len(idxs)))
    axes = np.atleast_2d(axes)
    for r, i in enumerate(idxs):
        x, y = test_ds[i]; img = (x*STD+MEAN).permute(1,2,0).numpy()
        axes[r,0].imshow(img); axes[r,0].set_title(f'True: {CLASSES[y]}', fontsize=8)
        for k, (name, g) in enumerate(cams.items()):
            cam, c, pr = g(x)
            axes[r,1+2*k].imshow(cam, cmap='jet'); axes[r,1+2*k].set_title(f'{name}\nCAM', fontsize=8)
            axes[r,2+2*k].imshow(img); axes[r,2+2*k].imshow(cam, cmap='jet', alpha=.5)
            axes[r,2+2*k].set_title(f'pred {CLASSES[c]} ({pr[c]:.2f})', fontsize=8)
        for a in axes[r]: a.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.savefig(OUT/fname, dpi=120); plt.show()

# one correctly-classified example per class (by ResNet-18's test predictions)
p_res, y_te = preds['ResNet-18']
idx = [int(np.where((y_te==k)&(p_res==k))[0][0]) for k in range(n) if ((y_te==k)&(p_res==k)).any()]
show_gradcam(idx, 'Grad-CAM, correct predictions (one per class)', 'gradcam_correct.png')

# misclassified examples
wrong = np.where(p_res != y_te)[0][:6]
if len(wrong): show_gradcam(list(map(int, wrong)), 'Grad-CAM, ResNet-18 errors', 'gradcam_errors.png')

## 8. Notes for the report
* **Imbalance:** this archive is balanced (~100 per class). Class-weighted loss is implemented, but its effect will be near zero here. For a real imbalance study, rebuild the data from the full WM-811K pickle (`LSWMD.pkl`, 172,950 maps, only ~14.8% labelled, "none" is ~85%).
* **Image quality:** the images are 32x32 JPEGs, so compression blurs the three discrete die states (background, pass, fail). Results may be a little lower than on the raw arrays.
* **Variance:** with ~135 test images, one image is ~0.7% accuracy. Report mean and std over several seeds (change `CFG['seed']`) or use k-fold CV before drawing conclusions about which model is better.
* **Reference paper:** the Radon transform and kernel-flip approach (Jeong et al., Sci. Rep. 2023) is an optional extension. Feeding Radon features into the same CNN would be a good third comparison.
* **Class definitions:** "near full" and "none" are kept here. The paper excludes them for the data-scarcity study. Set `CFG['drop_classes']` to compare.